# 3.0 · nnU-Net baseline on MSLesSeg (Kaggle)

Trains nnU-Net v2 (`3d_fullres`) on one cross-validation fold, using **our patient-level folds** (`splits_final.json`, built by `ms_seg/nnunet_data.py`).

**Before running (Kaggle notebook settings, right panel):**
1. *Add Input* → the Kaggle dataset `mslesseg-nnunet` (upload of `data/processed/kaggle/mslesseg_nnunet.zip`).
2. *Accelerator* → **GPU T4 x2** (not P100: recent PyTorch versions no longer support the P100's older architecture). nnU-Net uses one GPU.
3. *Internet* → **On** (to install nnU-Net).
4. Run with **Save Version → Save & Run All (Commit)**, so that it keeps running with the browser closed and `/kaggle/working` is saved.

Related issue: #10

## 1. Parameters
First run: a short smoke test (`nnUNetTrainer_5epochs`) to check that everything works and to measure the time per epoch. Then the real runs, one fold per session.

In [ ]:
FOLD = 0                           # 0-4
TRAINER = "nnUNetTrainer_5epochs"  # smoke test; later e.g. nnUNetTrainer_100epochs / _250epochs
CONFIG = "3d_fullres"
DATASET_ID = 101

## 2. Install nnU-Net and check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q nnunetv2
import subprocess, torch, nnunetv2
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU: set Accelerator = GPU T4 x2 in the notebook settings"

def run(cmd):
    """Run a shell command, print its output live and stop the notebook if it fails."""
    print("$", cmd)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"command failed (exit code {p.returncode}): {cmd}")

## 3. Locate the data and set the nnU-Net folders
`nnUNet_raw` is read from the (read-only) Kaggle input; preprocessed data go to `/kaggle/temp` (not saved) and results to `/kaggle/working` (saved with the notebook version).

In [ ]:
import os, glob, json, shutil
from pathlib import Path

ds_json = glob.glob("/kaggle/input/**/Dataset101_MSLesSeg/dataset.json", recursive=True)
assert ds_json, "dataset not found: add the mslesseg-nnunet dataset as input"
RAW = Path(ds_json[0]).parents[1]
SPLITS = next(Path("/kaggle/input").rglob("nnUNet_preprocessed/Dataset101_MSLesSeg/splits_final.json"))
PREP = Path("/kaggle/temp/nnUNet_preprocessed")
RES = Path("/kaggle/working/nnUNet_results")
for p in (PREP, RES):
    p.mkdir(parents=True, exist_ok=True)
os.environ.update(nnUNet_raw=str(RAW), nnUNet_preprocessed=str(PREP), nnUNet_results=str(RES),
                  nnUNet_n_proc_DA=str(os.cpu_count()),   # data-augmentation workers = CPUs (4 on Kaggle)
                  nnUNet_def_n_proc=str(os.cpu_count()))  # other nnU-Net workers (default 8 would exceed the CPUs)
print("raw:", RAW, "| training cases:", len(list((RAW / "Dataset101_MSLesSeg" / "labelsTr").glob("*.nii.gz"))))
splits = json.loads(SPLITS.read_text())
print("folds:", [(len(s["train"]), len(s["val"])) for s in splits])

## 4. Plan and preprocess
nnU-Net inspects the data (spacing, size, intensities), chooses the network and patch size, and preprocesses every case (cropping, z-score normalisation per image). Takes ~15–20 min.

In [ ]:
run(f"nnUNetv2_plan_and_preprocess -d {DATASET_ID} -c {CONFIG} --verify_dataset_integrity -npfp 4 -np 4")

## 5. Use our patient-level folds
nnU-Net would otherwise create random folds by case, which would put visits of the same patient in training and validation.

In [ ]:
dst = PREP / "Dataset101_MSLesSeg" / "splits_final.json"
shutil.copyfile(SPLITS, dst)
print("copied", dst, "| fold", FOLD, "val cases:", splits[FOLD]["val"])

## 6. Train

In [ ]:
run(f"nnUNetv2_train {DATASET_ID} {CONFIG} {FOLD} -tr {TRAINER} --npz")

## 7. Results of this fold
Validation Dice computed by nnU-Net (our own metrics, including LTPR/LFPR with 6-connectivity, are computed later with `ms_seg/metrics.py` on the saved predictions).

In [ ]:
from IPython.display import Image, display
fold_dir = RES / "Dataset101_MSLesSeg" / f"{TRAINER}__nnUNetPlans__{CONFIG}" / f"fold_{FOLD}"
summary = json.loads((fold_dir / "validation" / "summary.json").read_text())
print("mean validation Dice:", round(summary["foreground_mean"]["Dice"], 4))
display(Image(str(fold_dir / "progress.png")))
log = sorted(fold_dir.glob("training_log_*.txt"))[-1].read_text().splitlines()
times = [float(l.split("Epoch time:")[1].split("s")[0]) for l in log if "Epoch time" in l]
t = sum(times[1:]) / (len(times) - 1) if len(times) > 1 else times[0]  # skip the first (slower) epoch
print(f"time per epoch: {t:.0f} s -> 100 epochs = {100 * t / 3600:.1f} h, 250 epochs = {250 * t / 3600:.1f} h")

## 8. Pack the results for download
Download `results_fold{FOLD}.zip` from the *Output* tab of the saved version and unzip it into `models/nnunet/` of the project.

In [ ]:
shutil.make_archive(f"/kaggle/working/results_fold{FOLD}", "zip", RES)
print(os.listdir("/kaggle/working"))